In [2]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from pathlib import Path

# ----------------------------
# 0) LOAD
# ----------------------------
FILE_PATH = (
    Path.home()
    / "Documents"
    / "Master Degree in Sport Analytics"
    / "2. Management and Architecture of Sports Database"
    / "selenium wire"
    / "Result_SerieA_25_26"
    / "2_Bologna_Como_cmdro0qfo163kml8ooz371rtg.csv"
)

df = pd.read_csv(FILE_PATH)

# ----------------------------
# 1) HELPERS
# ----------------------------
def pick_col(df, candidates):
    cols_lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    for cand in candidates:
        for c in df.columns:
            if cand.lower() in c.lower():
                return c
    return None

# core columns (based on what you showed earlier in your project)
col_event   = pick_col(df, ["event"])
col_team    = pick_col(df, ["team_name", "team"])
col_player  = pick_col(df, ["player_name", "player"])
col_x       = pick_col(df, ["x"])
col_y       = pick_col(df, ["y"])
col_endx    = pick_col(df, ["end_x", "endx", "pass end x"])
col_endy    = pick_col(df, ["end_y", "endy", "pass end y"])
col_outcome = pick_col(df, ["outcome"])

# recipient candidates (if your export has one, we will use it!)
col_recipient = pick_col(df, [
    "recipient", "receiver", "pass_recipient", "pass_recipient_name",
    "related_player", "related_player_name", "to_player", "to_player_name"
])

# optional sort columns for fallback recipient
col_period  = pick_col(df, ["period_id", "period"])
col_min     = pick_col(df, ["time_min", "minute"])
col_sec     = pick_col(df, ["time_sec", "second"])
col_evid    = pick_col(df, ["event_id", "id"])

# ----------------------------
# 2) FILTER: BOLOGNA + PASS + SUCCESS
# ----------------------------
df_bol = df[df[col_team].astype(str).str.contains("Bologna", case=False, na=False)].copy()

# "Pass" vs "pass" safety
df_bol = df_bol[df_bol[col_event].astype(str).str.lower().eq("pass")].copy()

# success definition (your files look like outcome 1/0)
df_bol[col_outcome] = pd.to_numeric(df_bol[col_outcome], errors="coerce")
df_bol = df_bol[df_bol[col_outcome] == 1].copy()

# keep only rows with coords + player
need = [col_player, col_x, col_y, col_endx, col_endy]
df_bol = df_bol.dropna(subset=need).copy()

# ----------------------------
# 3) GET RECIPIENT
# ----------------------------
df_bol["passer"] = df_bol[col_player].astype(str)

if col_recipient is not None:
    df_bol["recipient"] = df_bol[col_recipient].astype(str)
else:
    # fallback: next Bologna event's player (approximation)
    sort_cols = [c for c in [col_period, col_min, col_sec, col_evid] if c is not None]
    if len(sort_cols) == 0:
        # last resort: keep original order
        sort_cols = []
    if len(sort_cols) > 0:
        df_bol = df_bol.sort_values(sort_cols).reset_index(drop=True)

    df_bol["recipient"] = df_bol["passer"].shift(-1)

# drop bad recipient rows
df_bol = df_bol.dropna(subset=["recipient"])
df_bol = df_bol[(df_bol["passer"].str.len() > 0) & (df_bol["recipient"].str.len() > 0)]
df_bol = df_bol[df_bol["passer"] != df_bol["recipient"]].copy()

# ----------------------------
# 3b) STARTERS vs SUBS (approx)
# ----------------------------
# We define "first appearance order" using available time columns.
sort_cols = [c for c in [col_period, col_min, col_sec, col_evid] if c is not None]
tmp = df_bol.copy()

if len(sort_cols) > 0:
    tmp = tmp.sort_values(sort_cols)
else:
    tmp = tmp.reset_index(drop=True)

first_seen = tmp.groupby("passer", as_index=False).head(1)
starters = first_seen["passer"].head(11).tolist()   # first 11 appearing = starters (approx)

# ----------------------------
# 3c) APPROX XI ON-FIELD (near end): keep subs + remaining starters
# ----------------------------
# build a time column for ordering (robust)
df_bol["_period"] = pd.to_numeric(df_bol[col_period], errors="coerce") if col_period else 1
df_bol["_min"]    = pd.to_numeric(df_bol[col_min], errors="coerce") if col_min else 0
df_bol["_sec"]    = pd.to_numeric(df_bol[col_sec], errors="coerce") if col_sec else 0

# A single sortable timeline value
df_bol["_t"] = df_bol["_period"] * 10_000 + df_bol["_min"] * 100 + df_bol["_sec"]

# First/last seen per player (using passer)
seen = (df_bol.groupby("passer", as_index=False)
        .agg(first_t=("_t", "min"), last_t=("_t", "max"))
        .sort_values("first_t"))

# subs = players not in starters
subs = [p for p in seen["passer"].tolist() if p not in starters]

# approximate "subbed off" = starters with earliest last_seen
k = min(len(subs), len(starters))
starters_last = seen[seen["passer"].isin(starters)].sort_values("last_t")
off_players = starters_last["passer"].head(k).tolist()

# on-field near end ≈ remaining starters + all subs
on_field_end = [p for p in starters if p not in off_players] + subs

# keep exactly 11: keep those who appear latest
seen_map_last = dict(zip(seen["passer"], seen["last_t"]))
on_field_end = sorted(on_field_end, key=lambda p: seen_map_last.get(p, -1), reverse=True)[:11]


# ----------------------------
# 4) NODES: average locations (like typical pass network)
# ----------------------------
df_bol[col_x] = pd.to_numeric(df_bol[col_x], errors="coerce")
df_bol[col_y] = pd.to_numeric(df_bol[col_y], errors="coerce")
df_bol = df_bol.dropna(subset=[col_x, col_y])

# average passer start location + pass count per passer
nodes = (df_bol.groupby("passer", as_index=False)
         .agg(x=(col_x, "mean"),
              y=(col_y, "mean"),
              passes=("passer", "size")))
nodes["role"] = np.where(nodes["passer"].isin(starters), "Starter", "Sub")

# ----------------------------
# 5) EDGES: pass counts between players
# ----------------------------
edges = (df_bol.groupby(["passer", "recipient"], as_index=False)
         .size()
         .rename(columns={"size": "count"}))

# ----------------------------
# 6) SCALE FOR PLOTTING (mplsoccer example-style)
# ----------------------------
# Node size scaling
min_node_size = 200
max_node_size = 1200
if nodes["passes"].max() == nodes["passes"].min():
    nodes["node_size"] = (min_node_size + max_node_size) / 2
else:
    nodes["node_size"] = (
        min_node_size
        + (nodes["passes"] - nodes["passes"].min())
        * (max_node_size - min_node_size)
        / (nodes["passes"].max() - nodes["passes"].min())
    )

# Edge width scaling
min_lw = 2.5
max_lw = 12.0
alpha = 0.7
if len(edges) == 0 or edges["count"].max() == edges["count"].min():
    edges["lw"] = (min_lw + max_lw) / 2
else:
    edges["lw"] = (
        min_lw
        + (edges["count"] - edges["count"].min())
        * (max_lw - min_lw)
        / (edges["count"].max() - edges["count"].min())
    )

# merge node coords onto edges
edges = edges.merge(nodes[["passer", "x", "y"]].rename(columns={"x": "x_start", "y": "y_start"}),
                    on="passer", how="left")
edges = edges.merge(nodes[["passer", "x", "y"]].rename(columns={"passer": "recipient", "x": "x_end", "y": "y_end"}),
                    on="recipient", how="left")
edges = edges.dropna(subset=["x_start", "y_start", "x_end", "y_end"])

# role mapping for edges
role_map = nodes.set_index("passer")["role"].to_dict()
edges["role_passer"] = edges["passer"].map(role_map)
edges["role_recipient"] = edges["recipient"].map(role_map)

# ----------------------------
# 6b) SPLIT VIEWS (XI / on-field after subs / all)
# ----------------------------
nodes_starters = nodes[nodes["passer"].isin(starters)].copy()
nodes_onfield  = nodes[nodes["passer"].isin(on_field_end)].copy()
nodes_all      = nodes.copy()

edges_starters = edges[edges["passer"].isin(starters) & edges["recipient"].isin(starters)].copy()
edges_onfield  = edges[edges["passer"].isin(on_field_end) & edges["recipient"].isin(on_field_end)].copy()
edges_all      = edges.copy()

# ----------------------------
# 7) PLOT (Plotly version with hover like the xT notebook)
# ----------------------------

def pitch_layout(fig, title):
    # Plotly pitch (0-100 coordinates)
    fig.update_xaxes(range=[0, 100], showgrid=False, zeroline=False, visible=False)
    # keep pitch aspect similar to mplsoccer (Opta pitch ~ 100x100 but visually rectangular)
    fig.update_yaxes(range=[0, 100], showgrid=False, zeroline=False, visible=False, scaleanchor="x", scaleratio=0.68)
    fig.update_yaxes(autorange=False)  # 0 bottom, 100 top

    fig.update_layout(
        title=title,
        width=1250,
        height=780,
        margin=dict(l=20, r=460, t=60, b=20),
        plot_bgcolor="black",
        paper_bgcolor="black",
        font=dict(color="white")
    )

    # Pitch lines (simple)
    line = dict(color="white", width=2)
    thin = dict(color="white", width=1)

    # Outline
    fig.add_shape(type="rect", x0=0, y0=0, x1=100, y1=100, line=line)
    # Halfway line
    fig.add_shape(type="line", x0=50, y0=0, x1=50, y1=100, line=thin)
    # Penalty areas
    fig.add_shape(type="rect", x0=0, y0=21, x1=17, y1=79, line=thin)
    fig.add_shape(type="rect", x0=83, y0=21, x1=100, y1=79, line=thin)
    # 6-yard boxes
    fig.add_shape(type="rect", x0=0, y0=36, x1=6, y1=64, line=thin)
    fig.add_shape(type="rect", x0=94, y0=36, x1=100, y1=64, line=thin)
    # Center circle (approx)
    fig.add_shape(
        type="circle",
        x0=50-9.15, y0=50-9.15*(100/68),
        x1=50+9.15, y1=50+9.15*(100/68),
        line=thin
    )
    return fig


# --- build Plotly figure ---
fig = go.Figure()

def add_network_traces(fig, edges_df, nodes_df, prefix, visible):
    start_idx = len(fig.data)

    # edges (one trace per connection)
    for _, r in edges_df.iterrows():
        fig.add_trace(go.Scatter(
            x=[r["x_start"], r["x_end"]],
            y=[r["y_start"], r["y_end"]],
            mode="lines",
            line=dict(color="#0F2235", width=float(r["lw"])),
            opacity=0.9,
            hoverinfo="text",
            text=f"{r['passer']} → {r['recipient']}<br>Passes: {int(r['count'])}",
            showlegend=False,
            visible=visible,
            name=f"{prefix}_edge"
        ))

    # nodes (legend OFF — we add a custom text legend with annotations)
    fig.add_trace(go.Scatter(
        x=nodes_df["x"],
        y=nodes_df["y"],
        mode="markers",
        marker=dict(
            size=np.sqrt(nodes_df["node_size"]) / 0.8,
            color="#8A1F33",
            line=dict(color="black", width=1),
            opacity=0.95
        ),
        hoverinfo="text",
        text=[f"{p}<br>Successful passes: {int(n)}" for p, n in zip(nodes_df["passer"], nodes_df["passes"])],
        name=f"{prefix} Players",
        visible=visible,
        showlegend=False
    ))

    # labels (legend OFF)
    fig.add_trace(go.Scatter(
        x=nodes_df["x"],
        y=nodes_df["y"],
        mode="text",
        text=nodes_df["passer"],
        textfont=dict(color="white", size=11),
        hoverinfo="skip",
        name=f"{prefix} Labels",
        visible=visible,
        showlegend=False
    ))

    end_idx = len(fig.data)
    return start_idx, end_idx  # [start, end)

# Add 3 groups and record their trace ranges
r_st  = add_network_traces(fig, edges_starters, nodes_starters, "Starters", visible=True)
r_sub = add_network_traces(fig, edges_onfield,  nodes_onfield,  "Substitutes", visible=False)
r_all = add_network_traces(fig, edges_all,      nodes_all,      "All", visible=False)

n_traces = len(fig.data)

def make_mask(rng):
    m = [False] * n_traces
    a, b = rng
    for i in range(a, b):
        m[i] = True
    return m

mask_starters = make_mask(r_st)
mask_subs     = make_mask(r_sub)
mask_all      = make_mask(r_all)

# Buttons (same color as the nodes)
BUTTON_RED = "#8A1F33"

fig.update_layout(
    # Keep the plot geometry stable when clicking buttons
    autosize=False,
    uirevision="passing_network_static",

    updatemenus=[
        dict(
            type="buttons",
            direction="down",
            x=1.14,
            y=0.995,
            xanchor="left",
            yanchor="top",
            showactive=True,
            bgcolor="rgba(0,0,0,0.35)",
            bordercolor=BUTTON_RED,
            borderwidth=1,
            font=dict(color=BUTTON_RED, size=14),
            buttons=[
                dict(label="Starters (XI)", method="update", args=[{"visible": mask_starters}]),
                dict(label="Substitutes",  method="update", args=[{"visible": mask_subs}]),
                dict(label="All",          method="update", args=[{"visible": mask_all}]),
            ],
        )
    ],

    # Hide Plotly legend entirely (we add a custom text legend so there are no marker "dots")
    showlegend=False,

    # Custom legend (text only, no dots / Aa)
    annotations=[]
)

fig = pitch_layout(fig, "Passing Network - Bologna (vs. Como) - MatchDay 2")
fig.show()

print("Successful pass rows in df_bol (after basic filters):", len(df_bol))
print("Nodes:", len(nodes), "| Edges:", len(edges))
print("Recipient column used:", col_recipient if col_recipient is not None else "FALLBACK (next-event player)")

Successful pass rows in df_bol (after basic filters): 186
Nodes: 16 | Edges: 87
Recipient column used: FALLBACK (next-event player)
